## Cell 1 — Load Collected Webcam Images

In [7]:
import sys

from pathlib import Path

In [8]:
# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Define image directory
if IN_COLAB:
    drive.mount('/content/drive')

    # Google Drive folder
    IMAGE_DIR = Path('/content/drive/MyDrive/webcam_images')

else:
    # notebooks/ -> project root

    PROJECT_ROOT = Path.cwd().parent

    if str(PROJECT_ROOT) not in sys.path:

        sys.path.append(str(PROJECT_ROOT))

    IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "images"

image_files = list(IMAGE_DIR.glob("*.jpg"))

print("Running in Colab:", IN_COLAB)

print("Project root:", PROJECT_ROOT if not IN_COLAB else "Colab")

print("Image directory:", IMAGE_DIR)

print("Number of images:", len(image_files))

Running in Colab: False
Project root: /Users/qingxuan/Desktop/webcam-pm25-toolbox
Image directory: /Users/qingxuan/Desktop/webcam-pm25-toolbox/data/raw/images
Number of images: 220


## Cell 2 — ROI Extraction

In [9]:
import importlib
import src.roi_viewer as roi_viewer

importlib.reload(roi_viewer)

roi_controls = roi_viewer.build_roi_viewer(IMAGE_DIR)

Output()

In [10]:
import json

selected_roi = {
    "top": roi_controls["top"].value,
    "left": roi_controls["left"].value,
    "height": roi_controls["height"].value,
    "width": roi_controls["width"].value
}

ROI_JSON = PROJECT_ROOT / "config" / "roi.json"
ROI_JSON.parent.mkdir(parents=True, exist_ok=True)

with open(ROI_JSON, "w") as f:
    json.dump(selected_roi, f, indent=4)

print("Selected ROI:")
print(f"top = {selected_roi['top']}")
print(f"left = {selected_roi['left']}")
print(f"height = {selected_roi['height']}")
print(f"width = {selected_roi['width']}")
print(f"mode = {roi_controls['mode'].value}")

print("\nROI saved to:")
print(ROI_JSON)

Selected ROI:
top = 160
left = 116
height = 380
width = 515
mode = RGB

ROI saved to:
/Users/qingxuan/Desktop/webcam-pm25-toolbox/config/roi.json


## Cell 3 — Image Feature Extraction

In [16]:
import importlib
import pandas as pd
import src.image_features as image_features

importlib.reload(image_features)

rows, skipped_files = image_features.extract_image_features(
    image_dir=PROJECT_ROOT / "data" / "raw" / "images",
    output_csv=PROJECT_ROOT / "data" / "interim" / "image_features.csv",
    roi=selected_roi
)

df = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "image_features.csv")

print(f"Done. Wrote {len(rows)} rows.")
print(f"Skipped files: {len(skipped_files)}")

df.head()

Done. Wrote 220 rows.
Skipped files: 0


,datetime,R_roi,G_roi,B_roi,S_mean,B_R_ratio,contrast,image_path
0,2026-03-01 03:00:00,86.927317,103.467195,122.422105,0.290022,1.408327,19.237125,data/raw/images/20260301-0400.jpg
1,2026-03-01 04:00:00,86.576546,103.945815,123.102785,0.295734,1.421895,14.599862,data/raw/images/20260301-0500.jpg
2,2026-03-01 05:00:00,88.301727,103.318487,121.760818,0.273552,1.378918,13.725783,data/raw/images/20260301-0600.jpg
3,2026-03-01 06:00:00,97.634921,127.311369,157.803475,0.381719,1.616261,17.681084,data/raw/images/20260301-0700.jpg
4,2026-03-01 07:00:00,112.163654,144.063168,172.775825,0.352873,1.540390,21.162996,data/raw/images/20260301-0800.jpg
